In [0]:
# ============================================================================
# NOTEBOOK 3: GOLD LAYER - BUSINESS-READY AGGREGATIONS
# ============================================================================
# Purpose: Create aggregated tables for analytics and business intelligence

# Gold Layer - Business Analytics Tables
# Aggregate and enrich data for dashboards and reporting

import pyspark.sql.functions as F
from pyspark.sql.window import Window
from datetime import datetime

catalog = "workspace"
schema = "ecommerce_dq"

print("=" * 60)
print("GOLD LAYER - BUSINESS AGGREGATIONS")
print("=" * 60)



In [0]:
# Cell 1: Load Silver Tables
silver_products = spark.table(f"{catalog}.{schema}.silver_products")
silver_customers = spark.table(f"{catalog}.{schema}.silver_customers")
silver_orders = spark.table(f"{catalog}.{schema}.silver_orders")
silver_events = spark.table(f"{catalog}.{schema}.silver_events")

print("✅ Loaded all Silver tables")

In [0]:


# Cell 2: GOLD TABLE 1 - Daily Sales Summary
def create_gold_daily_sales():
    """
    Daily aggregation of orders with key metrics
    """
    df = silver_orders.groupBy(
        F.col("order_date").alias("sales_date"),
        "year",
        "month"
    ).agg(
        F.count("id").alias("total_orders"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_order_value"),
        F.min("total_amount").alias("min_order_amount"),
        F.max("total_amount").alias("max_order_amount"),
        F.stddev("total_amount").alias("revenue_stddev")
    ).orderBy(F.col("sales_date").desc())
    
    # Add calculations
    df = df.withColumn("created_at", F.current_timestamp())
    df = df.withColumn("gold_version", F.lit("v1.0"))
    
    table_name = f"{catalog}.{schema}.gold_daily_sales"
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f"✅ Gold Daily Sales: {df.count()} rows")
    return df

In [0]:
# Cell 3: GOLD TABLE 2 - Customer Lifetime Value (CORRIGIDO)
def create_gold_customer_ltv():
    """
    Customer analytics with lifetime value and order metrics
    """
    
    # Join customers with orders - usar alias para clareza
    customer_orders = silver_customers.alias("c").join(
        silver_orders.alias("o"),
        F.col("c.id") == F.col("o.customer_id"),
        "left"
    )
    
    # Aggregate by customer
    df = customer_orders.groupBy(
        F.col("c.id").alias("customer_id"),
        F.col("c.full_name"),
        F.col("c.email"),
        F.col("c.city")
    ).agg(
        F.count(F.col("o.id")).alias("total_orders"),
        F.sum(F.col("o.total_amount")).alias("lifetime_value"),
        F.avg(F.col("o.total_amount")).alias("avg_order_value"),
        F.max(F.col("o.order_date")).alias("last_order_date"),
        F.min(F.col("o.order_date")).alias("first_order_date"),
        F.datediff(F.max(F.col("o.order_date")), F.min(F.col("o.order_date"))).alias("customer_tenure_days")
    )
    
    # Calculate RFM
    df = df.withColumn(
        "recency_days",
        F.datediff(F.current_date(), F.col("last_order_date"))
    )
    
    # Segment customers
    df = df.withColumn(
        "customer_segment",
        F.when(F.col("lifetime_value") > 1000, "VIP")
         .when(F.col("lifetime_value") > 500, "Premium")
         .when(F.col("lifetime_value") > 100, "Regular")
         .otherwise("New")
    )
    
    df = df.withColumn("created_at", F.current_timestamp())
    
    table_name = f"{catalog}.{schema}.gold_customer_ltv"
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f"✅ Gold Customer LTV: {df.count()} rows")
    return df

In [0]:

# Cell 4: GOLD TABLE 3 - Product Performance
def create_gold_product_performance():
    """
    Product analytics with sales and rating metrics
    """
    
    # Join products with orders
    product_orders = silver_products.join(
        silver_orders,
        silver_orders.id == silver_products.id,
        "left"
    )
    
    # Aggregate by product
    df = silver_products.groupBy(
        "id",
        "product_name",
        "category",
        "price",
        "avg_rating"
    ).agg(
        F.count(F.col("id")).alias("units_sold"),
        F.sum(F.col("price")).alias("total_revenue"),
        F.avg(F.col("price")).alias("avg_selling_price")
    ).na.fill(0)
    
    # Add metrics
    df = df.withColumn(
        "revenue_per_unit",
        F.round(F.col("total_revenue") / F.greatest(F.col("units_sold"), F.lit(1)), 2)
    )
    
    # Rank products
    window_spec = Window.partitionBy("category").orderBy(F.col("total_revenue").desc())
    df = df.withColumn("rank_in_category", F.row_number().over(window_spec))
    
    # Performance tier
    df = df.withColumn(
        "performance_tier",
        F.when(F.col("avg_rating") >= 4.5, "Excellent")
         .when(F.col("avg_rating") >= 4.0, "Good")
         .when(F.col("avg_rating") >= 3.5, "Average")
         .otherwise("Poor")
    )
    
    df = df.withColumn("created_at", F.current_timestamp())
    
    table_name = f"{catalog}.{schema}.gold_product_performance"
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f"✅ Gold Product Performance: {df.count()} rows")
    return df


In [0]:
# Cell 5: GOLD TABLE 4 - Event Analytics
def create_gold_event_analytics():
    """
    User behavior and engagement metrics
    """
    df = silver_events.groupBy(
        "event_date",
        "event_type",
        "device"
    ).agg(
        F.count("event_id").alias("event_count"),
        F.countDistinct("user_id").alias("unique_users"),
        F.count_distinct("event_id").alias("total_events")
    )
    
    # Calculate engagement rate
    df = df.withColumn(
        "engagement_per_user",
        F.round(F.col("event_count") / F.col("unique_users"), 2)
    )
    
    # Event type summary
    event_summary = silver_events.groupBy("event_type").agg(
        F.count("event_id").alias("total_events"),
        F.countDistinct("user_id").alias("unique_users")
    )
    
    # Device summary
    device_summary = silver_events.groupBy("device").agg(
        F.count("event_id").alias("total_events"),
        F.countDistinct("user_id").alias("unique_users")
    )
    
    table_name = f"{catalog}.{schema}.gold_event_analytics"
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f"✅ Gold Event Analytics: {df.count()} rows")
    
    # Save summaries too
    event_summary_table = f"{catalog}.{schema}.gold_event_summary"
    event_summary.write.format("delta").mode("overwrite").saveAsTable(event_summary_table)
    
    device_summary_table = f"{catalog}.{schema}.gold_device_summary"
    device_summary.write.format("delta").mode("overwrite").saveAsTable(device_summary_table)
    
    return df


In [0]:

# Cell 6: GOLD TABLE 5 - Geographic Analysis
def create_gold_geographic_analysis():
    """
    Sales and customer metrics by city
    """
    
    # Join customers with orders - use aliases to disambiguate columns
    customer_sales = silver_customers.alias("c").join(
        silver_orders.alias("o"),
        F.col("c.id") == F.col("o.customer_id"),
        "left"
    )
    
    # Aggregate by city
    df = customer_sales.groupBy("city").agg(
        F.countDistinct(F.col("c.id")).alias("unique_customers"),
        F.count(F.col("o.id")).alias("total_orders"),
        F.sum(F.col("o.total_amount")).alias("total_revenue"),
        F.avg(F.col("o.total_amount")).alias("avg_order_value"),
        F.max(F.col("o.order_date")).alias("last_order_date")
    ).na.fill(0)
    
    # Add metrics
    df = df.withColumn(
        "avg_customer_value",
        F.round(F.col("total_revenue") / F.greatest(F.col("unique_customers"), F.lit(1)), 2)
    )
    
    # Market size ranking
    window_spec = Window.orderBy(F.col("total_revenue").desc())
    df = df.withColumn("market_rank", F.row_number().over(window_spec))
    
    df = df.withColumn("created_at", F.current_timestamp())
    
    table_name = f"{catalog}.{schema}.gold_geographic_analysis"
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f"✅ Gold Geographic Analysis: {df.count()} rows")
    return df


In [0]:

# Cell 7: Run All Gold Transformations
print("\n" + "=" * 60)
print("Creating Gold Tables...")
print("=" * 60 + "\n")

gold_daily_sales = create_gold_daily_sales()
gold_customer_ltv = create_gold_customer_ltv()
gold_product_perf = create_gold_product_performance()
gold_events = create_gold_event_analytics()
gold_geo = create_gold_geographic_analysis()

print("\n" + "=" * 60)
print("✅ GOLD LAYER COMPLETE!")
print("=" * 60)


In [0]:

# Cell 8: Summary & Data Quality
print("\n📊 GOLD TABLES SUMMARY:\n")

tables = [
    "gold_daily_sales",
    "gold_customer_ltv", 
    "gold_product_performance",
    "gold_event_analytics",
    "gold_geographic_analysis"
]

for table in tables:
    count = spark.table(f"{catalog}.{schema}.{table}").count()
    print(f"  ✅ {table}: {count} rows")


In [0]:

# Cell 9: Show Sample Data
print("\n" + "=" * 60)
print("SAMPLE DATA - Daily Sales")
print("=" * 60)
display(spark.table(f"{catalog}.{schema}.gold_daily_sales").limit(5))

print("\n" + "=" * 60)
print("SAMPLE DATA - Customer LTV")
print("=" * 60)
display(spark.table(f"{catalog}.{schema}.gold_customer_ltv").limit(5))

print("\n" + "=" * 60)
print("SAMPLE DATA - Product Performance")
print("=" * 60)
display(spark.table(f"{catalog}.{schema}.gold_product_performance").limit(5))

print("\n" + "=" * 60)
print("SAMPLE DATA - Geographic Analysis")
print("=" * 60)
display(spark.table(f"{catalog}.{schema}.gold_geographic_analysis").limit(5))